In [4]:

import os
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    FunctionTransformer
)
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
os.makedirs("../outputs", exist_ok=True)

print("LOADING ADULT DATASET")
adult = fetch_openml(
    "adult",
    version=2,
    as_frame=True
)

df = adult.frame.copy()
df = df.replace("?", np.nan)

# CREATE TARGET
df["target"] = df["class"].map({
    "<=50K": 0,
    ">50K": 1
}).astype(int)

X = df.drop(columns=["class", "target"])
y = df["target"]
# ENGINEERED FEATURE FUNCTION

def create_engineered_features(data):

    data = data.copy()

    # Age bucket
    data["age_bucket"] = pd.cut(
        data["age"],
        bins=[0, 25, 35, 45, 55, 65, np.inf],
        labels=[
            "Young",
            "Early_Career",
            "Mid_Career",
            "Experienced",
            "Senior",
            "Older"
        ],
        include_lowest=True
    )

    # Hours-per-week bucket
    data["hours_bucket"] = pd.cut(
        data["hours-per-week"],
        bins=[0, 30, 40, 50, 60, np.inf],
        labels=[
            "Part_Time",
            "Standard",
            "Overtime",
            "High_Hours",
            "Very_High_Hours"
        ],
        include_lowest=True
    )

    # Capital gain flag
    data["capital_gain_flag"] = (
        data["capital-gain"] > 0
    ).astype(int)

    # Log capital gain
    data["log_capital_gain"] = np.log1p(
        data["capital-gain"]
    )

    # Higher education flag
    data["higher_education"] = (
        data["education-num"] >= 13
    ).astype(int)

    # Education × working hours
    data["education_hours_interaction"] = (
        data["education-num"]
        * data["hours-per-week"]
    )

    # Capital loss flag
    data["capital_loss_flag"] = (
        data["capital-loss"] > 0
    ).astype(int)

    # Age × working hours
    data["age_hours_interaction"] = (
        data["age"]
        * data["hours-per-week"]
    )

    return data
# APPLY FEATURE ENGINEERING THROUGH FUNCTIONTRANSFORMER
feature_engineering = FunctionTransformer(
    create_engineered_features,
    validate=False
)

# FEATURE LISTS

numeric_features = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week",

    # Engineered numeric features
    "capital_gain_flag",
    "log_capital_gain",
    "higher_education",
    "education_hours_interaction",
    "capital_loss_flag",
    "age_hours_interaction"
]

categorical_features = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",

    # Engineered categorical features
    "age_bucket",
    "hours_bucket"
]
#  NUMERIC PREPROCESSING

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])
# CATEGORICAL PREPROCESSING

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])
# COLUMN TRANSFORMER

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])
# COMPLETE LOGISTIC REGRESSION PIPELINE

logistic_pipeline = Pipeline([
    (
        "feature_engineering",
        feature_engineering
    ),

    (
        "preprocessor",
        preprocessor
    ),

    (
        "classifier",
        LogisticRegression(
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])
# COMPLETE RANDOM FOREST PIPELINE

random_forest_pipeline = Pipeline([
    (
        "feature_engineering",
        feature_engineering
    ),

    (
        "preprocessor",
        preprocessor
    ),

    (
        "classifier",
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    )
])
# TRAIN / TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("\nTraining samples:", len(X_train))
print("Testing samples :", len(X_test))

# FIT PIPELINES
print("\nTRAINING LOGISTIC REGRESSION PIPELINE\n")
logistic_pipeline.fit(
    X_train,
    y_train
)

print("Logistic Regression trained successfully!")

print("\nTRAINING RANDOM FOREST PIPELINE\n")
random_forest_pipeline.fit(
    X_train,
    y_train
)
print("Random Forest trained successfully!")
# CHECK TRANSFORMED DATA
X_train_transformed = (
    logistic_pipeline
    .named_steps["preprocessor"]
    .transform(
        logistic_pipeline
        .named_steps["feature_engineering"]
        .transform(X_train)
    )
)


print("\n---PIPELINE CHECK---\n")


print(
    "Original training features:",
    X_train.shape[1]
)

print(
    "Features after engineering:",
    create_engineered_features(X_train).shape[1]
)

print(
    "Features after preprocessing:",
    X_train_transformed.shape[1]
)
#  VERIFY ENGINEERED FEATURES
engineered_sample = create_engineered_features(
    X_train.head()
)

print("\n---ENGINEERED FEATURE CHECK---\n")

print(
    engineered_sample[
        [
            "age_bucket",
            "hours_bucket",
            "capital_gain_flag",
            "log_capital_gain",
            "higher_education",
            "education_hours_interaction",
            "capital_loss_flag",
            "age_hours_interaction"
        ]
    ]
)

import joblib

joblib.dump(
    logistic_pipeline,
    "../outputs/day3_logistic_pipeline.pkl"
)

joblib.dump(
    random_forest_pipeline,
    "../outputs/day3_random_forest_pipeline.pkl"
)




LOADING ADULT DATASET

Training samples: 39073
Testing samples : 9769

TRAINING LOGISTIC REGRESSION PIPELINE

Logistic Regression trained successfully!

TRAINING RANDOM FOREST PIPELINE

Random Forest trained successfully!

---PIPELINE CHECK---

Original training features: 14
Features after engineering: 22
Features after preprocessing: 122

---ENGINEERED FEATURE CHECK---

         age_bucket hours_bucket  capital_gain_flag  log_capital_gain  \
34342         Older    Part_Time                  0               0.0   
18559         Young    Part_Time                  0               0.0   
12477  Early_Career     Standard                  0               0.0   
560      Mid_Career     Standard                  0               0.0   
3427   Early_Career     Standard                  0               0.0   

       higher_education  education_hours_interaction  capital_loss_flag  \
34342                 0                          153                  0   
18559                 0              

['../outputs/day3_random_forest_pipeline.pkl']